# Symbolic Unconditioned Music Generation
**CSE 153/253 — Assignment 2**

## 1. Setup

In [11]:
!pip install pretty_midi

import pretty_midi
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import torch

## 2. Tokenizer

In [12]:
# vocabulary constants
NOTE_ON_OFFSET    = 0
NOTE_OFF_OFFSET   = 128
TIME_SHIFT_OFFSET = 256
VELOCITY_OFFSET   = 356

VOCAB_SIZE        = 388
N_TIME_SHIFT_BINS = 100
N_VELOCITY_BINS   = 32
TIME_STEP_MS      = 10
MAX_SHIFT_MS      = N_TIME_SHIFT_BINS * TIME_STEP_MS

def velocity_to_bin(velocity):
    return min(velocity // 4, N_VELOCITY_BINS - 1)

def bin_to_velocity(bin_idx):
    return bin_idx * 4 + 2

In [13]:
def midi_to_tokens(midi_path):
    midi = pretty_midi.PrettyMIDI(str(midi_path))

    # Step 1: collect raw events
    raw_events = []
    for instrument in midi.instruments:
        for note in instrument.notes:
            vbin = velocity_to_bin(note.velocity)
            raw_events.append((note.start, 'VELOCITY', vbin))
            raw_events.append((note.start, 'NOTE_ON',  note.pitch))
            raw_events.append((note.end,   'NOTE_OFF', note.pitch))

    # Step 2: sort by time; for ties, VELOCITY comes before NOTE_ON before NOTE_OFF
    type_order = {'VELOCITY': 0, 'NOTE_ON': 1, 'NOTE_OFF': 2}
    raw_events.sort(key=lambda e: (e[0], type_order[e[1]]))

    # Step 3: convert to tokens, inserting TIME_SHIFT tokens when the clock advances
    tokens = []
    current_time = 0.0  # seconds

    for event_time, event_type, value in raw_events:
        delta_ms = int(round((event_time - current_time) * 1000))

        # emit as many TIME_SHIFT tokens as needed to cover the gap
        while delta_ms > 0:
            shift = min(delta_ms, MAX_SHIFT_MS)
            n_steps = max(1, int(round(shift / TIME_STEP_MS)))
            n_steps = min(n_steps, N_TIME_SHIFT_BINS)
            tokens.append(TIME_SHIFT_OFFSET + n_steps - 1)
            delta_ms -= n_steps * TIME_STEP_MS

        current_time = event_time

        if event_type == 'NOTE_ON':
            tokens.append(NOTE_ON_OFFSET + value)
        elif event_type == 'NOTE_OFF':
            tokens.append(NOTE_OFF_OFFSET + value)
        elif event_type == 'VELOCITY':
            tokens.append(VELOCITY_OFFSET + value)

    return tokens

In [14]:
def tokens_to_midi(tokens, output_path):
    midi  = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano

    current_time     = 0.0
    current_velocity = bin_to_velocity(16)  # default: medium velocity
    open_notes       = {}  # pitch → (start_time, velocity)

    for token in tokens:
        if NOTE_ON_OFFSET <= token < NOTE_OFF_OFFSET:
            pitch = token - NOTE_ON_OFFSET
            open_notes[pitch] = (current_time, current_velocity)

        elif NOTE_OFF_OFFSET <= token < TIME_SHIFT_OFFSET:
            pitch = token - NOTE_OFF_OFFSET
            if pitch in open_notes:
                start, vel = open_notes.pop(pitch)
                end = max(current_time, start + 0.01)  # ensure non-zero duration
                piano.notes.append(pretty_midi.Note(
                    velocity=vel, pitch=pitch, start=start, end=end
                ))

        elif TIME_SHIFT_OFFSET <= token < VELOCITY_OFFSET:
            n_steps = token - TIME_SHIFT_OFFSET + 1
            current_time += n_steps * TIME_STEP_MS / 1000.0

        elif VELOCITY_OFFSET <= token < VOCAB_SIZE:
            current_velocity = bin_to_velocity(token - VELOCITY_OFFSET)

    # close any notes still open at the end
    for pitch, (start, vel) in open_notes.items():
        piano.notes.append(pretty_midi.Note(
            velocity=vel, pitch=pitch, start=start, end=max(current_time, start + 0.01)
        ))

    midi.instruments.append(piano)
    midi.write(str(output_path))
    print(f"Saved: {output_path}")

## 3. Sanity Check

In [15]:
# Part 1: create a simple hand-crafted MIDI: C4 (pitch=60) then E4 (pitch=64)
test_midi = pretty_midi.PrettyMIDI()
piano     = pretty_midi.Instrument(program=0)
piano.notes.append(pretty_midi.Note(velocity=80, pitch=60, start=0.0, end=0.5))
piano.notes.append(pretty_midi.Note(velocity=60, pitch=64, start=0.5, end=1.0))
test_midi.instruments.append(piano)
test_midi.write("test_input.mid")

# Tokenize it
tokens = midi_to_tokens("test_input.mid")

# Print tokens with human-readable labels
def decode_token(t):
    if t < NOTE_OFF_OFFSET:
        return f"NOTE_ON({t})"
    elif t < TIME_SHIFT_OFFSET:
        return f"NOTE_OFF({t - NOTE_OFF_OFFSET})"
    elif t < VELOCITY_OFFSET:
        return f"TIME_SHIFT({(t - TIME_SHIFT_OFFSET + 1) * TIME_STEP_MS}ms)"
    else:
        return f"VELOCITY(bin={t - VELOCITY_OFFSET})"

print(f"Tokens ({len(tokens)} total):")
for t in tokens:
    print(f"  {t:3d}  →  {decode_token(t)}")

# Part 2: round-trip check
tokens_to_midi(tokens, "test_roundtrip.mid")
tokens_rt = midi_to_tokens("test_roundtrip.mid")
print(f"\nRound-trip match: {tokens == tokens_rt}")

Tokens (8 total):
  376  →  VELOCITY(bin=20)
   60  →  NOTE_ON(60)
  305  →  TIME_SHIFT(500ms)
  371  →  VELOCITY(bin=15)
   64  →  NOTE_ON(64)
  188  →  NOTE_OFF(60)
  305  →  TIME_SHIFT(500ms)
  192  →  NOTE_OFF(64)
Saved: test_roundtrip.mid

Round-trip match: True


## 4. Data Loading

In [16]:
import csv

DATASET_DIR = Path("maestro-v3.0.0")

# Load the CSV metadata to get the official train/validation/test splits
train_files = []
val_files   = []

with open(DATASET_DIR / "maestro-v3.0.0.csv", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        midi_path = DATASET_DIR / row["midi_filename"]
        if row["split"] == "train":
            train_files.append(midi_path)
        elif row["split"] == "validation":
            val_files.append(midi_path)

print(f"Train files : {len(train_files)}")
print(f"Val files   : {len(val_files)}")

Train files : 962
Val files   : 137


In [17]:
def tokenize_dataset(file_list, output_path):
    all_tokens = []
    for i, path in enumerate(file_list):
        try:
            tokens = midi_to_tokens(path)
            all_tokens.extend(tokens)
        except Exception as e:
            print(f"Skipping {path.name}: {e}")
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(file_list)} files processed")

    arr = np.array(all_tokens, dtype=np.int16)
    np.save(output_path, arr)
    print(f"Saved {len(arr):,} tokens to {output_path}")
    return arr

print("Tokenizing train set...")
train_tokens = tokenize_dataset(train_files, "train_tokens.npy")

print("\nTokenizing validation set...")
val_tokens = tokenize_dataset(val_files, "val_tokens.npy")

Tokenizing train set...
  100/962 files processed
  200/962 files processed
  300/962 files processed
  400/962 files processed
  500/962 files processed
  600/962 files processed
  700/962 files processed
  800/962 files processed
  900/962 files processed
Saved 31,423,621 tokens to train_tokens.npy

Tokenizing validation set...
  100/137 files processed
Saved 3,554,013 tokens to val_tokens.npy


In [18]:
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 512

class MusicDataset(Dataset):
    def __init__(self, tokens):
        self.tokens = torch.tensor(tokens.astype(np.int64))

    def __len__(self):
        # each sample needs SEQ_LEN input tokens + 1 target token
        return len(self.tokens) - SEQ_LEN

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + SEQ_LEN]          # input
        y = self.tokens[idx + 1 : idx + SEQ_LEN + 1]  # target (shifted by 1)
        return x, y

train_dataset = MusicDataset(train_tokens)
val_dataset   = MusicDataset(val_tokens)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train samples : {len(train_dataset):,}")
print(f"Val samples   : {len(val_dataset):,}")
print(f"Train batches : {len(train_loader):,}")

Train samples : 31,423,109
Val samples   : 3,553,501
Train batches : 981,973


## 5. Model

In [19]:
import torch.nn as nn

class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_size, num_layers,
                                 batch_first=True, dropout=dropout)
        self.fc        = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x, hidden = self.lstm(self.embedding(x), hidden)
        return self.fc(x), hidden

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MusicLSTM(
    vocab_size   = VOCAB_SIZE,
    embed_dim    = 128,
    hidden_size  = 512,
    num_layers   = 2,
    dropout      = 0.2,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Using device: cpu
Total parameters: 3,664,772


In [20]:
import torch.nn.functional as F

LEARNING_RATE = 0.001
MAX_STEPS     = 50   # change to a small number (e.g. 10) to test on CPU
VAL_EVERY     = 1000    # validate every N steps
SAVE_EVERY    = 5000    # save checkpoint every N steps

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def evaluate(model, loader, max_batches=50):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
            total_loss += loss.item()
    model.train()
    return total_loss / min(max_batches, len(loader))

train_iter = iter(train_loader)
step       = 0
train_losses = []
val_losses   = []

model.train()
while step < MAX_STEPS:
    # get next batch, reset iterator if exhausted
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)

    x, y = x.to(device), y.to(device)

    optimizer.zero_grad()
    logits, _ = model(x)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_losses.append(loss.item())
    step += 1

    if step % VAL_EVERY == 0:
        val_loss = evaluate(model, val_loader)
        val_losses.append((step, val_loss))
        print(f"Step {step:6d} | train loss: {loss.item():.4f} | val loss: {val_loss:.4f}")

    if step % SAVE_EVERY == 0:
        torch.save(model.state_dict(), f"checkpoint_step{step}.pt")
        print(f"  → checkpoint saved")

print("Training complete.")

Training complete.


## 7. Generation

In [21]:
def generate(model, seed_tokens, n_tokens=1000, temperature=1.0):
    model.eval()
    tokens  = list(seed_tokens)
    hidden  = None

    # warm up the model's hidden state with the seed
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        _, hidden = model(x, hidden)

    # generate one token at a time
    current_token = torch.tensor([[tokens[-1]]], dtype=torch.long).to(device)
    with torch.no_grad():
        for _ in range(n_tokens):
            logits, hidden = model(current_token, hidden)
            logits = logits[:, -1, :] / temperature       # scale by temperature
            probs  = torch.softmax(logits, dim=-1)         # convert to probabilities
            next_token = torch.multinomial(probs, 1).item()  # sample
            tokens.append(next_token)
            current_token = torch.tensor([[next_token]], dtype=torch.long).to(device)

    return tokens[len(seed_tokens):]  # return only the generated part

# use the first 100 tokens of a real piece as seed
seed = train_tokens[:100].astype(np.int64).tolist()

generated = generate(model, seed, n_tokens=2000, temperature=1.0)
tokens_to_midi(generated, "symbolic_unconditioned.mid")
print(f"Generated {len(generated)} tokens")

Saved: symbolic_unconditioned.mid
Generated 2000 tokens


## 8. Evaluation